# 07 — The Fisher–Rao Metric

Answer the question `02/06` left open — which ruler is natural on the simplex? — with a criterion, not a preference: two distributions are close when *samples cannot tell them apart*. Make that precise by hand for a coin (the length of a step is the number of standard errors it moves the sample mean), derive the Fisher information $I(\theta) = 1/\theta(1-\theta)$ twice — from the score, and as the Hessian of relative entropy — and see it diverge at the boundary: every step near a corner is expensive (the corners themselves stay at a finite distance — `02/08` computes it — because the blow-up is only like $1/\sqrt p$). Extend it to three outcomes, $ds^2 = \sum_i dp_i^2/p_i$, draw the *unit circles* of this metric across the triangle and watch them squeeze against the edges, and finally meet the theorem that makes it *the* choice: Chentsov's — the only metric that does not change when you split an outcome into sub-outcomes and never grows when you merge outcomes. The geodesics and the closed-form distance are `02/08`'s.

**Prerequisites:** `02/06_the_probability_simplex` (the triangle, barycentric coordinates, the four rulers and their disagreement near the boundary), `02/02_kl_divergence` (relative entropy, its Hessian at $q = p$), `02/03_laws_of_relative_entropy` (the data-processing inequality, of which merging outcomes is a case, and the equality case of the log-sum inequality), `01/11_expectation_values` (shot noise: the standard error $\sqrt{p(1-p)/N}$ of an estimated probability), `01/15` (the Bhattacharyya overlap behind the Hellinger distance).

**What you'll be able to do**
- Turn "samples cannot tell $\theta$ from $\theta + \delta$" into a length: $\delta$ divided by the standard error $\sqrt{\theta(1-\theta)/N}$ — and read off $I(\theta) = 1/\theta(1-\theta)$ as the metric of a coin, derived by hand from the score $\partial_\theta\log p$.
- Derive $D(p_\theta\|p_{\theta+\delta}) \approx \tfrac12 I(\theta)\delta^2$ (nats) and $\mathrm{Hel}^2 \approx \tfrac18 I\delta^2$ by Taylor expansion — the two rulers of `02/06` that counted surprise are the same ruler up to second order — and check both numerically.
- Write the metric on three outcomes, $g = \mathrm{diag}(1/p_i)$ on tangent vectors with $\sum_i dp_i = 0$, compute the Fisher length of `02/06`'s two $0.02$-shifts, and draw the metric's unit circles (Tissot ellipses) across the triangle.
- State Chentsov's theorem and *check its two halves by hand*: splitting an outcome in fixed proportions leaves every Fisher length unchanged (Euclidean length changes), and merging outcomes never increases it, with equality exactly when the merged ratios agree — the same condition as `02/03`'s log-sum equality.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.geometry.info_geometry import bernoulli_fisher, multinomial_fisher
from qot_course.infotheory.classical import kl_divergence

rng = np.random.default_rng(0)   # coin samples for the distinguishability experiment of section 1
viz.use_course_style()

## 1. Distinguishability, by hand: a step counted in standard errors

Take a coin with $P(\text{heads}) = \theta$ and a second coin with $\theta + \delta$. How many flips do you need to tell them apart? From $N$ flips you estimate $\hat\theta$ with standard error $\sigma_N = \sqrt{\theta(1-\theta)/N}$ (`01/11`). The two coins are distinguishable when the step $\delta$ is a few standard errors wide, $\delta \gtrsim \sigma_N$ — so the natural *length* of the step is not $\delta$ but

$$
\frac{\delta}{\sigma_N} = \delta\,\sqrt{\frac{N}{\theta(1-\theta)}} = \sqrt{N}\;\sqrt{I(\theta)}\;\delta, \qquad I(\theta) = \frac{1}{\theta(1-\theta)} .
$$

The same $\delta$ is a short step near $\theta = \tfrac12$ and a long one near $\theta = 0$ or $1$: a $0.01$ change is invisible on a fair coin but glaring on a coin that almost never lands heads. Equivalently, the number of flips needed to resolve the step at a fixed confidence is $N \approx z^2/(I(\theta)\delta^2)$: **the metric is the price, in samples, of a unit of Euclidean distance**, and $\sqrt{I}\,\delta$ is the step measured in units the data can feel.

**The same number from the score.** Fisher's definition (1925; the object Rao turned into a metric in 1945) is the variance of the *score*, the sensitivity of the log-likelihood to the parameter:
$$
I(\theta) = \mathbb E_{p_\theta}\!\Big[\Big(\frac{\partial\log p_\theta(x)}{\partial\theta}\Big)^2\Big] .
$$
For the coin, $\log p_\theta(\text{heads}) = \log\theta$ and $\log p_\theta(\text{tails}) = \log(1-\theta)$, so the score is $1/\theta$ on heads and $-1/(1-\theta)$ on tails, and its mean square is
$$
\theta\cdot\frac{1}{\theta^2} + (1-\theta)\cdot\frac{1}{(1-\theta)^2} = \frac1\theta + \frac1{1-\theta} = \frac{1}{\theta(1-\theta)} .
$$
(Its mean is $\theta\cdot\tfrac1\theta - (1-\theta)\cdot\tfrac1{1-\theta} = 0$, always: the score has zero mean because $\sum_x p_\theta(x) = 1$ for every $\theta$. That fact will do work in section 2.) Let's measure distinguishability directly — simulate $N$ flips of each of two coins a step $\delta = 0.01$ apart, at $\theta = 0.5$ and at $\theta = 0.97$, and count how many standard errors (of one coin's mean) separate the two sample means — then draw $I(\theta)$.


In [ ]:
def separation_in_se(theta, delta, N, trials=4000):
    # sample means of two coins a step delta apart; how many standard errors apart are they?
    a = rng.binomial(N, theta, size=trials) / N
    b = rng.binomial(N, theta + delta, size=trials) / N
    return (b.mean() - a.mean()) / a.std(ddof=1)

delta, N = 0.01, 2000
for theta in [0.5, 0.8, 0.97]:
    sep = separation_in_se(theta, delta, N)
    print(f"theta = {theta:.2f}: the step {delta} is {sep:5.2f} standard errors wide at N = {N}   (prediction sqrt(N I) delta = {np.sqrt(N * bernoulli_fisher(theta)) * delta:5.2f};  I = {bernoulli_fisher(theta):6.2f})")
for theta in [0.5, 0.97]:
    N_needed = 4 / (bernoulli_fisher(theta) * delta**2)       # z = 2 standard errors
    print(f"theta = {theta:.2f}: flips needed to make the step {delta} two standard errors wide: {N_needed:8.0f}")

thetas = np.linspace(0.005, 0.995, 400)
fisher = np.array([bernoulli_fisher(t) for t in thetas])
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2), gridspec_kw={"wspace": 0.28})
ax = axes[0]
ax.plot(thetas, fisher, color=viz.SOURCE_COLOR, lw=2.4)
ax.axvline(0.5, color=viz.TARGET_COLOR, ls="--", lw=1, label="theta = 1/2: I = 4, the flattest point")
ax.set_yscale("log"); ax.set_xlabel("coin parameter theta"); ax.set_ylabel("Fisher information I(theta)  (log scale)")
ax.set_title("I(theta) = 1 / theta(1 - theta)", pad=10); ax.legend(loc="upper center")
ax = axes[1]
ax.plot(thetas, np.sqrt(fisher) * delta, color=viz.SOURCE_COLOR, lw=2.4, label=f"Fisher length of a step {delta}:  sqrt(I) delta")
ax.axhline(delta, color=COLORS["muted"], ls=":", lw=1.2, label=f"its Euclidean length {delta} (arbitrary scale, for reference)")
ax.set_yscale("log"); ax.set_xlabel("coin parameter theta"); ax.set_ylabel("length of the same step  (log scale)")
ax.set_title("Fisher length of one Euclidean step of 0.01", pad=10); ax.legend(loc="upper center")
plt.show()

**Read the output and the figure.** A step of $0.01$ on a fair coin is $0.9$ standard errors wide at $N = 2000$ — invisible; the same step at $\theta = 0.97$ is $2.6$ standard errors wide — a clear separation — and the measured separations match $\sqrt{N I(\theta)}\,\delta$. To make the step two standard errors wide you need about $10{,}000$ flips at $\theta = \tfrac12$ and about $1{,}200$ at $\theta = 0.97$: the metric is the price in samples, and it is eight-to-nine times cheaper near the edge. Left: $I(\theta)$ is smallest at the fair coin ($I = 4$) and climbs without bound at both ends — on a log axis, straight up. Right: the same Euclidean step of $0.01$ has a Fisher length that grows without bound toward the boundary — near the boundary every step is expensive in this metric, not because the probabilities are far, but because a tiny change of a near-zero probability is a large statistical event. (Expensive, not unreachable. What you have built is a **metric**: a rule giving the squared length $ds^2 = I(\theta)\,d\theta^2$ of every infinitesimal step at every point; the length of a *path* is $\int ds$. Here $ds = d\theta/\sqrt{\theta(1-\theta)}$ blows up only like $1/\sqrt\theta$ at the corner, so the integral converges — the corner sits at a finite distance, which `02/08` names.) This is what the Hellinger and KL level sets of `02/06` were flattening against.


## 2. The same metric, twice more: the Hessian of relative entropy, and Hellinger

`02/06` found that KL and Hellinger agreed near a point and disagreed from Euclidean and TV — "statistical rulers count surprise". Here is why: **both are the Fisher metric to second order.** Expand the relative entropy (in nats) of a distribution $p$ to a nearby $p + dp$ (with $\sum_i dp_i = 0$, so the sum stays $1$):
$$
D(p\,\|\,p + dp) = -\sum_i p_i\ln\Big(1 + \frac{dp_i}{p_i}\Big) = -\sum_i p_i\Big(\frac{dp_i}{p_i} - \frac{dp_i^2}{2p_i^2} + \dots\Big) = -\underbrace{\sum_i dp_i}_{0} + \frac12\sum_i\frac{dp_i^2}{p_i} + O(dp^3) .
$$
The first-order term vanishes exactly because the total probability is conserved — the same "zero-mean score" fact as in section 1 — and what is left is *half the Fisher quadratic form*: $D(p\|p + dp) \approx \tfrac12\,ds^2$ with $ds^2 = \sum_i dp_i^2/p_i$. For a coin, $ds^2 = d\theta^2/\theta + d\theta^2/(1-\theta) = I(\theta)\,d\theta^2$ — the same $I$. (The expansion of $D(p + dp\|p)$ gives the same leading term: the asymmetry of $D$ is third order.) And Hellinger: $\mathrm{Hel}^2 = 1 - \sum_i\sqrt{p_i(p_i + dp_i)} = 1 - \sum_i p_i\sqrt{1 + dp_i/p_i} = 1 - \sum_i p_i\big(1 + \tfrac{dp_i}{2p_i} - \tfrac{dp_i^2}{8p_i^2} + \dots\big) = \tfrac18\,ds^2 + O(dp^3)$. Relative entropy, Hellinger, and the sample-counting argument of section 1 are three faces of one quadratic form. Let's check both expansions on a coin and on a three-outcome $p$, for steps shrinking to zero, and watch the ratio to $ds^2$ settle on $\tfrac12$ and $\tfrac18$.


In [ ]:
def fisher_ds2(p, dp):
    return float(np.sum(np.asarray(dp) ** 2 / np.asarray(p)))

def hellinger2(p, q):
    return float(1.0 - np.sum(np.sqrt(np.asarray(p) * np.asarray(q))))

p3 = np.array([0.7, 0.2, 0.1]); v3 = np.array([0.5, -0.2, -0.3])     # a tangent direction: entries sum to 0
print(f"{'step h':>8s} {'D(p||p+h v)/ds^2 (coin, theta=0.3)':>36s} {'Hel^2/ds^2 (coin)':>19s} {'D/ds^2 (3 outcomes)':>21s} {'Hel^2/ds^2 (3 out.)':>20s}")
for hstep in [0.1, 0.03, 0.01, 0.003, 0.001]:
    theta = 0.3; pc = np.array([theta, 1 - theta]); vc = np.array([1.0, -1.0])
    D_c = kl_divergence(pc, pc + hstep * vc, base=np.e); ds2_c = fisher_ds2(pc, hstep * vc)
    D_3 = kl_divergence(p3, p3 + hstep * v3, base=np.e); ds2_3 = fisher_ds2(p3, hstep * v3)
    print(f"{hstep:8.3f} {D_c / ds2_c:36.4f} {hellinger2(pc, pc + hstep * vc) / ds2_c:19.4f} {D_3 / ds2_3:21.4f} {hellinger2(p3, p3 + hstep * v3) / ds2_3:20.4f}")
print(f"limits: 1/2 = {0.5}, 1/8 = {0.125};  and for the coin ds^2 = I(theta) dtheta^2: {fisher_ds2(pc, 0.01 * vc):.6f} = {bernoulli_fisher(0.3) * 0.01**2:.6f}")
# the asymmetry is third order: D(p||q) - D(q||p) shrinks like h^3
for hstep in [0.1, 0.01, 0.001]:
    q = p3 + hstep * v3
    print(f"h = {hstep}: D(p||q) - D(q||p) = {kl_divergence(p3, q, base=np.e) - kl_divergence(q, p3, base=np.e):.2e}  (ratio to h^3: {(kl_divergence(p3, q, base=np.e) - kl_divergence(q, p3, base=np.e)) / hstep**3:.3f})")

**Read the output.** As the step shrinks, $D/ds^2$ settles on $0.5$ and $\mathrm{Hel}^2/ds^2$ on $0.125$, on the coin and on three outcomes alike — the Taylor expansions, confirmed to within $10^{-3}$ at $h = 0.001$ — and the coin's $ds^2$ is exactly $I(\theta)\,d\theta^2$. The asymmetry of relative entropy shrinks like $h^3$ (its ratio to $h^3$ settles on a constant as $h \to 0$), so near a point $D$ *is* symmetric and quadratic: relative entropy is the Fisher metric plus corrections, which is why `02/02`'s two landscapes looked alike near their shared zero. Euclidean distance has the quadratic form $\sum_i dp_i^2$, blind to $p$, and total variation is not even quadratic — it is $\tfrac12\|dp\|_1$, polyhedral, the hexagons of `02/06` — neither comes from a metric that counts samples, and that is the whole difference between the two families of rulers in `02/06`.


## 3. Three outcomes: the metric on the triangle, and its unit circles

For a categorical distribution with $n$ outcomes, parametrised by its own entries, the score for outcome $x$ with respect to $p_j$ is $\partial\log p_x/\partial p_j = \delta_{xj}/p_j$, so
$$
g_{jk} = \sum_x p_x\,\frac{\delta_{xj}}{p_j}\frac{\delta_{xk}}{p_k} = \frac{\delta_{jk}}{p_j}, \qquad g = \mathrm{diag}\Big(\frac1{p_1}, \dots, \frac1{p_n}\Big), \qquad ds^2 = \sum_i \frac{dp_i^2}{p_i} ,
$$
to be applied to *tangent* vectors $dp$ with $\sum_i dp_i = 0$ — the directions that stay on the simplex (the helper `multinomial_fisher` returns this diagonal matrix; the constraint is yours to respect). Strictly, $n$ free entries are one too many for a statistical model; the honest statement is that in *any* parametrisation $\theta$ the score variance is $I_{ab} = \sum_x \partial_a p_x\,\partial_b p_x/p_x$, so the length of a tangent step is always $\sum_x dp_x^2/p_x$ — in the reduced coordinates $(p_1, p_2)$ the matrix is not diagonal, but it is the same quadratic form, and "$\mathrm{diag}(1/p)$ restricted to the tangent plane" is the coordinate-free way to write it. The metric is **diagonal but not constant**: the price of moving outcome $i$ is $1/p_i$, steep where $p_i$ is small. Take `02/06`'s two moves of $0.02$ of mass from $p = (0.5, 0.48, 0.02)$: between the two common outcomes, $ds^2 = 0.02^2\,(1/0.5 + 1/0.48) = 0.0016$; taking the rare outcome's $0.02$ away, $ds^2 = 0.02^2\,(1/0.02 + 1/0.5) = 0.0208$ — about thirteen times larger for the same Euclidean step. One honesty note: $ds^2$ is the *infinitesimal* price at $p$, and this second move removes *all* of outcome $3$ — a $100\%$ relative change, far outside the linear regime — so the tangent price undershoots the true length: the exact Fisher–Rao lengths (`02/08`'s closed form) are $0.040$ and $0.285$, a ratio of $7$ — the same $7$ that `02/06`'s Hellinger found, as section 2 predicts. Here we compare the *tangent* prices, $0.0016$ vs $0.0208$; a step that crossed the edge would have infinite tangent price.

The cleanest picture of a metric is its **unit circle** at each point: the set of tangent vectors of Fisher length $r$ (a small fixed $r$), drawn on the triangle. Where the metric is large the circle is small, and where it is larger in one direction than another the circle is an ellipse (cartographers call these Tissot's indicatrices). Let's compute them by hand: choose an orthonormal basis $e_1 = (1,-1,0)/\sqrt2$, $e_2 = (1,1,-2)/\sqrt6$ of the tangent plane (two unit vectors, orthogonal to each other and to $(1,1,1)$ — so their combinations are exactly the $dp$ with $\sum_i dp_i = 0$), write the $2\times2$ matrix $G = E^\top\mathrm{diag}(1/p)E$, and draw $\{v : v^\top G v = r^2\}$ at a grid of points. $G$ is symmetric and positive, so in its eigenbasis $v^\top G v = \lambda_1 v_1^2 + \lambda_2 v_2^2$ (eigenvalues of a Hermitian matrix, `01/13`): the set is an ellipse with semi-axes $r/\sqrt{\lambda_k}$ along the eigenvectors — large eigenvalue, short axis.


In [ ]:
V = np.array([[0.0, 0.0], [1.0, 0.0], [0.5, np.sqrt(3) / 2]])      # the triangle of 02/06
E = np.stack([np.array([1, -1, 0]) / np.sqrt(2), np.array([1, 1, -2]) / np.sqrt(6)], axis=1)   # orthonormal basis of the tangent plane (3 x 2)

def tissot(p, r=0.1, m=80):
    G = E.T @ multinomial_fisher(p) @ E                        # the 2x2 Fisher metric in tangent coordinates
    w, U = np.linalg.eigh(G)
    ang = np.linspace(0, 2 * np.pi, m)
    circle = np.stack([np.cos(ang), np.sin(ang)])             # unit circle in the eigenbasis ...
    v = U @ (circle / np.sqrt(w)[:, None]) * r                 # ... scaled to Fisher length r
    dp = E @ v                                                 # back to 3-D tangent vectors (sum to 0)
    return (p[None, :] + dp.T) @ V, w                          # ellipse points in the plane, and the two eigenvalues

p_edge = np.array([0.5, 0.48, 0.02])
for name, dp in [("between the common outcomes", np.array([-0.02, 0.02, 0.0])), ("remove the rare outcome's mass", np.array([0.0, 0.02, -0.02]))]:
    print(f"{name:32s}: Euclidean^2 = {np.sum(dp**2):.4f}   Fisher ds^2 = {fisher_ds2(p_edge, dp):.4f}   ratio ds^2/Euclid^2 = {fisher_ds2(p_edge, dp) / np.sum(dp**2):6.1f}")

fig, ax = plt.subplots(figsize=(8.5, 7.2))
tri = np.vstack([V, V[:1]]); ax.plot(tri[:, 0], tri[:, 1], color=COLORS["muted"], lw=1.5)   # the triangle of 02/06, by hand
for (x, y), lab, dx, dy in zip(V, [r"$\delta_1=(1,0,0)$", r"$\delta_2=(0,1,0)$", r"$\delta_3=(0,0,1)$"], [-0.07, 0.07, 0.0], [-0.045, -0.045, 0.045]):
    ax.text(x + dx, y + dy, lab, ha="center", va="center", fontsize=11, color=COLORS["text"])
ax.set_aspect("equal"); ax.axis("off")
grid_pts = []
for i in range(1, 11):
    for j in range(1, 11 - i):
        a_, b_ = i / 11, j / 11
        grid_pts.append(np.array([a_, b_, 1 - a_ - b_]))
for p in grid_pts:
    pts, w = tissot(p)
    ax.plot(pts[:, 0], pts[:, 1], color=viz.SOURCE_COLOR, lw=1.6)
    ax.scatter(*(p @ V), s=6, color=COLORS["text"], zorder=4)
ax.set_title("Unit circles of the Fisher metric (equal Fisher length r = 0.1) across the triangle", pad=26)
plt.show()
w_centre = tissot(np.array([1/3, 1/3, 1/3]))[1]; w_edge = tissot(np.array([0.45, 0.45, 0.10]))[1]
print(f"metric eigenvalues at the centre: {np.round(w_centre, 2)} (isotropic, = 3 = 1/p);  near the bottom edge (0.45, 0.45, 0.10): {np.round(w_edge, 2)} -- {w_edge.max() / w_edge.min():.1f} x stiffer across the edge than along it")

**Read the output and the figure.** The two $0.02$-moves of `02/06` have the same Euclidean length and tangent Fisher prices in the ratio $13$ (exact lengths: ratio $7$, the Hellinger number of `02/06` — the linearisation is only a guide for a move that empties an outcome): the metric charges $1/p_i$ per unit of squared displacement of outcome $i$, and the rare outcome is expensive. On the triangle, the unit circles are round and largest at the centre, where $g = 3I_2$ and every direction costs the same, and they shrink and flatten into ellipses toward the edges — squeezed *across* the nearby edge (moving the small probability) and relatively free *along* it (trading mass between the two large probabilities): near $(0.45, 0.45, 0.10)$ the metric is $3.3$ times stiffer across than along. The corners, where two probabilities vanish, are stiff in every direction — the ellipses would shrink to points there, although, as the coin computation showed, the total length to a corner stays finite. The picture is the answer to `02/06`'s question in one figure — the triangle is not flat, and because the metric is cheaper in the middle, the shortest paths that `02/08` computes bend away from the boundary.


## 4. Why this metric: Chentsov's theorem, checked by hand

There are many Riemannian metrics on the triangle. What singles this one out is a criterion, not a taste: a statistical distance should not depend on *how finely you happen to label the outcomes*. Two operations make that precise (a third, *relabelling* — permuting the outcomes — is free: $\sum_i dp_i^2/p_i$ is a sum over outcomes and does not care about their order).

- **Splitting** (a Markov embedding): replace outcome $1$ by two sub-outcomes in fixed proportions $\lambda, 1 - \lambda$, so $p = (p_1, p_2, p_3) \mapsto p' = (\lambda p_1, (1-\lambda)p_1, p_2, p_3)$ and every tangent vector $dp \mapsto dp' = (\lambda\,dp_1, (1-\lambda)\,dp_1, dp_2, dp_3)$. The finer description carries no information about $p$ that the coarse one did not (the split is a fixed coin that knows nothing about $p$: the coarse description already carries all the information — a *sufficient statistic*), so a statistical metric must give $dp'$ the same length as $dp$. Check by hand: $\frac{(\lambda dp_1)^2}{\lambda p_1} + \frac{((1-\lambda)dp_1)^2}{(1-\lambda)p_1} = \frac{dp_1^2}{p_1}(\lambda + 1 - \lambda) = \frac{dp_1^2}{p_1}$ — **exactly invariant**, for every $\lambda$. The Euclidean length is not: $(\lambda dp_1)^2 + ((1-\lambda)dp_1)^2 < dp_1^2$.
- **Merging** (coarse-graining, `02/03` §3): lump outcomes $1$ and $2$ into one, $p \mapsto (p_1 + p_2, p_3)$, $dp \mapsto (dp_1 + dp_2, dp_3)$. Information can be lost, never gained, so the length must not increase: $\frac{(dp_1 + dp_2)^2}{p_1 + p_2} \le \frac{dp_1^2}{p_1} + \frac{dp_2^2}{p_2}$ — the inequality $(a + b)^2 \le (\frac{a^2}{s} + \frac{b^2}{t})(s + t)$ with $a = dp_1, b = dp_2, s = p_1, t = p_2$, which is $(a\sqrt{t/s} - b\sqrt{s/t})^2 \ge 0$ expanded (Cauchy–Schwarz) — with equality iff $dp_1/p_1 = dp_2/p_2$: merging costs nothing exactly when the two outcomes were being moved *in proportion* — the same condition as the equality case of `02/03`'s log-sum inequality, which is no coincidence (section 2: $D \approx \tfrac12 ds^2$).

**Chentsov's theorem** (1972; short proof and extension to unnormalised measures, Campbell 1986; general sample spaces, Ay, Jost, Lê & Schwachhöfer 2015): *the Fisher metric is, up to a constant factor, the only Riemannian metric — defined on the simplices of all sizes at once — that is invariant under splitting;* monotonicity under merging is the complementary property checked above. The ruler that counts samples is forced. Let's verify both halves numerically on random points and directions — and watch the Euclidean length fail the first one.


In [ ]:
def split_outcome(p, dp, lam):
    return np.concatenate([[lam * p[0], (1 - lam) * p[0]], p[1:]]), np.concatenate([[lam * dp[0], (1 - lam) * dp[0]], dp[1:]])

def merge_first_two(p, dp):
    return np.concatenate([[p[0] + p[1]], p[2:]]), np.concatenate([[dp[0] + dp[1]], dp[2:]])

worst_split, worst_merge = 0.0, -np.inf
for _ in range(2000):
    p = rng.dirichlet(np.ones(3)); dp = rng.normal(size=3); dp -= dp.mean(); dp *= 0.01   # a random tangent direction
    lam = rng.uniform(0.05, 0.95)
    ps, dps = split_outcome(p, dp, lam); pm, dpm = merge_first_two(p, dp)
    worst_split = max(worst_split, abs(fisher_ds2(ps, dps) - fisher_ds2(p, dp)))
    worst_merge = max(worst_merge, fisher_ds2(pm, dpm) - fisher_ds2(p, dp))
print(f"splitting outcome 1 in random proportions: max |ds'^2 - ds^2| over 2000 trials = {worst_split:.1e}   (exactly invariant)")
print(f"merging outcomes 1 and 2:                  max (ds'^2 - ds^2) over 2000 trials  = {worst_merge:.2e}   (never positive)")
p = np.array([0.5, 0.3, 0.2]); dp = np.array([0.02, -0.015, -0.005]); ps, dps = split_outcome(p, dp, 0.3)
print(f"one example, p = {p}, dp = {dp}: Fisher ds^2 = {fisher_ds2(p, dp):.6f} -> after a 30/70 split {fisher_ds2(ps, dps):.6f};  Euclidean |dp|^2 = {np.sum(dp**2):.6f} -> {np.sum(dps**2):.6f} (changed)")
dp_prop = np.array([0.02, 0.012, -0.032])        # dp_1/p_1 = dp_2/p_2 = 0.04: moved in proportion
pm, dpm = merge_first_two(p, dp_prop)
print(f"merging two outcomes moved IN PROPORTION (dp_i/p_i equal): ds^2 = {fisher_ds2(p, dp_prop):.6f} -> {fisher_ds2(pm, dpm):.6f}  (no loss: the equality case)")
pm, dpm = merge_first_two(p, dp)
print(f"merging two outcomes moved out of proportion:              ds^2 = {fisher_ds2(p, dp):.6f} -> {fisher_ds2(pm, dpm):.6f}  (length lost)")

**Read the output.** Over two thousand random points, directions and split proportions, the Fisher length is unchanged by splitting to $10^{-16}$ — exact, as the one-line computation said — while the Euclidean length of the same step changes; and merging never increases it. The two merges of the last lines are the equality case and its failure: move two outcomes in proportion and lumping them loses nothing; move them out of proportion and length is lost. A metric that passes both tests is a metric that measures *information about $p$* and nothing about the labelling — and Chentsov's theorem says there is only one. That is the answer to `02/06`: the natural ruler on the simplex is the one that counts samples, it is forced by invariance, and it makes the triangle curved.


## 5. Where the metric goes

- **Geodesics and the closed form** (`02/08`): a metric gives shortest paths and a distance along them. For this metric both have a closed form, reached by a change of variable that turns the triangle into a piece of a sphere — `02/08` does it; today you know *why* the paths will bend away from the edges.
- **Cramér–Rao**: the metric also bounds estimation. For any unbiased estimator $\hat\theta$ from $N$ samples, $\mathrm{Var}(\hat\theta) \ge 1/(N I(\theta))$. For the coin the sample mean has variance $\theta(1-\theta)/N = 1/(N I)$ exactly — it *attains* the bound, which is why `01/11`'s standard error is what it is (exercise 4).
- **The quantum Fisher information** (`02/13`): replace $p$ by a density matrix $\rho$ and ask the same question — how well can measurements tell $\rho$ from $\rho + d\rho$? — and Chentsov's uniqueness *breaks*: there is a whole family of monotone metrics (Petz 1996), and the smallest is the **Bures metric**, which on diagonal states reduces to $\tfrac14\sum_i dp_i^2/p_i$ — today's metric, scaled. The dictionary row below records the promise.
- **A second geometry** (`02/09`, module 03): everything today is *vertical* — it compares masses bin by bin and is invariant under relabelling (Chentsov's operations never look at which bin is next to which). When the bins have a geometry of their own, a second ruler appears, and it measures something this one cannot.


## Your turn

1. **The flattest coin.** Confirm `bernoulli_fisher(0.5) == 4` and that it is the minimum over $\theta$ by plotting $I$ on a linear axis; then explain in one sentence, using section 1, why the *most uncertain* coin is the *least sensitive* to a change of $\theta$. Compute the number of flips needed to resolve $\delta = 0.005$ at two standard errors for $\theta \in \{0.5, 0.9, 0.99\}$.
2. **Stiffest coordinate, and an ellipse of your own.** With `multinomial_fisher([0.7, 0.2, 0.1])`, which coordinate is stiffest? Draw the Fisher unit circle at that point with `tissot` and at the centre, and give the ratio of their areas (the area of the ellipse is $\pi r^2/\sqrt{\det G}$ — check numerically).
3. **The Hessian in the other direction.** Section 2 expanded $D(p\|p + dp)$. Expand $D(p + dp\|p)$ by hand to second order and show the leading term is the same $\tfrac12\sum_i dp_i^2/p_i$; then verify numerically on three outcomes that $\big(D(p\|q) + D(q\|p)\big)/ds^2 \to 1$ and that the *difference* $D(p\|p+dp) - D(p+dp\|p)$ is $-\tfrac16\sum_i dp_i^3/p_i^2$ to leading order — check that the printed ratio $0.44$ in section 2 is exactly that number for `p3, v3`.
4. **Harder: Cramér–Rao, and the bound a biased estimator can beat.** (a) For the coin, estimate $\theta = 0.3$ from $N = 50$ flips with the sample mean, $5000$ times, and compare its variance with $1/(N I(\theta))$. (b) Now use the shrinkage estimator $\hat\theta' = (k + 1)/(N + 2)$ ($k$ heads; the posterior mean of a uniform prior): compute its variance *and* its bias; show the variance is below the Cramér–Rao bound and say why that is allowed (state the biased Cramér–Rao bound $\mathrm{Var} \ge (1 + b'(\theta))^2/(N I)$ and check it). (c) (independent of (a)–(b)) Prove the merging inequality of section 4 from Cauchy–Schwarz for $m$ outcomes merged into one, and show that the chain of two merges is again a merge — so the Fisher length is monotone under *every* coarse-graining, not only pairwise ones.


## What you built

- You turned "samples cannot tell them apart" into a length — a step divided by the standard error it moves — and derived $I(\theta) = 1/\theta(1-\theta)$ by hand from the score, then measured it: the same $0.01$ step is $0.9$ standard errors at a fair coin and $2.6$ at $\theta = 0.97$, and the price in flips differs eightfold.
- You derived $D(p\|p + dp) \approx \tfrac12\sum_i dp_i^2/p_i$ and $\mathrm{Hel}^2 \approx \tfrac18\sum_i dp_i^2/p_i$ by Taylor expansion and watched the ratios settle on $\tfrac12$ and $\tfrac18$: the two rulers of `02/06` that counted surprise are the Fisher metric to second order, and the asymmetry of $D$ is third order.
- You wrote the metric on three outcomes, $g = \mathrm{diag}(1/p_i)$ on the tangent plane, priced `02/06`'s two moves ($13\times$ in tangent price, $7\times$ in exact length), and drew its unit circles across the triangle — round at the centre, squeezed against the edges, vanishing at the corners.
- You checked Chentsov's two invariances by hand and numerically — splitting leaves every Fisher length unchanged (Euclidean fails), merging never increases it, equality iff moved in proportion — and stated the theorem that makes this metric the only one.
- You also carry a number into `02/08`: in this metric no two points of the simplex, not even two corners, are more than $\pi$ apart.
- You can now say what Chentsov's criterion becomes for density matrices: every metric of Petz's family carries it over, in that none of them ever grows when a quantum channel acts on the states.
- You can name the quantum version of the Cramér–Rao bound: it is built on the quantum Fisher information, and it is how the precision of a quantum sensor is counted.

The triangle is curved, its boundary is expensive to approach, and you know why the ruler that says so is forced. `02/08` walks its shortest paths.


**New in the dictionary.** The complete table lives in [the course dictionary](../dictionary.ipynb) (`notebooks/dictionary.ipynb`).

| Classical | Quantum |
| --- | --- |
| Fisher information metric on the simplex, $ds^2 = \sum_i dp_i^2/p_i$ | quantum Fisher information: a family of monotone metrics, the smallest being the Bures metric (`02/13`) |

## What's next

A curved space has curved shortest paths. In `02/08_fisher_rao_geodesics` we find them — and the distance along them — in closed form, through a change of variable that turns the triangle into a piece of a sphere; the geodesics bow away from the boundary, exactly as today's unit circles predict, and the Bhattacharyya overlap of `01/15` turns out to be the cosine of half the distance.


## References

- R. A. Fisher, *Theory of statistical estimation*, Math. Proc. Cambridge Philos. Soc. **22**, 700–725 (1925) — the information as the variance of the score. doi:10.1017/S0305004100009580
- C. R. Rao, *Information and accuracy attainable in the estimation of statistical parameters*, Bull. Calcutta Math. Soc. **37**, 81–91 (1945) — the Fisher information as a Riemannian metric, and the Cramér–Rao bound.
- N. N. Chentsov (Čencov), *Statistical Decision Rules and Optimal Inference*, Nauka (1972); Engl. transl. Translations of Mathematical Monographs **53**, AMS (1982) — uniqueness of the Fisher metric under Markov morphisms.
- L. L. Campbell, *An extended Čencov characterization of the information metric*, Proc. Amer. Math. Soc. **98**, 135–141 (1986) — the statement on the simplex used here. doi:10.1090/S0002-9939-1986-0848890-5
- N. Ay, J. Jost, H. V. Lê & L. Schwachhöfer, *Information geometry and sufficient statistics*, Probab. Theory Relat. Fields **162**, 327–364 (2015) — Chentsov's theorem in full generality. doi:10.1007/s00440-014-0574-8
- S.-i. Amari, *Information Geometry and Its Applications*, Springer (2016) — ch. 1 (divergence → metric, $D \approx \tfrac12 ds^2$), ch. 3 (invariance, information monotonicity, Chentsov's theorem). doi:10.1007/978-4-431-55978-8
- T. M. Cover & J. A. Thomas, *Elements of Information Theory*, 2nd ed., Wiley (2006) — §11.10 (Fisher information and the Cramér–Rao inequality). doi:10.1002/047174882X
- D. Petz, *Monotone metrics on matrix spaces*, Linear Algebra Appl. **244**, 81–96 (1996) — the quantum family of monotone metrics, of which Bures (`02/13`) is the smallest. doi:10.1016/0024-3795(94)00211-8

**Previous:** `notebooks/02_InformationAndGeometry/06_the_probability_simplex.ipynb` · **Next:** `notebooks/02_InformationAndGeometry/08_fisher_rao_geodesics.ipynb`
